In [0]:
spark.conf.set(
    "fs.azure.account.key.azsynapsestudy.dfs.core.windows.net",
    ""
)

In [0]:
EQ_gold_raw_df = spark.read.format("Delta").option("inferschema", "True").load("abfss://study@azsynapsestudy.dfs.core.windows.net/ADB/silver")

In [0]:
EQ_gold_raw_df.show(2)

+-------+---+--------------------+-------------+-------------+----+--------------------+--------------------+----+----+----+-----+--------+-------+---+---+--------+-------+--------------------+---+----------+----+---+-------+---------------+--------------------+-------------+---------+---------+---------+----------+--------------------+--------------------+
|   type|mag|               place|         time|      updated|  tz|                 url|              detail|felt| cdi| mmi|alert|  status|tsunami|sig|net|    code|sources|               types|nst|      dmin| rms|gap|magType|properties_type|               title|geometry_type| latitude|longitude|    depth|        id|           date_time|   Updated_date_time|
+-------+---+--------------------+-------------+-------------+----+--------------------+--------------------+----+----+----+-----+--------+-------+---+---+--------+-------+--------------------+---+----------+----+---+-------+---------------+--------------------+-------------+----

In [0]:
#Dim_type

dim_properties_type_raw=EQ_gold_raw_df.select('properties_type').distinct()
dim_properties_type_raw.show()
dim_properties_type_raw.count()

+----------------+
| properties_type|
+----------------+
|       Ice Quake|
|       Explosion|
|      Earthquake|
|    Quarry Blast|
|Mining Explosion|
|     Other Event|
|       Landslide|
+----------------+



7

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os

def handle_dimension(EQ_gold_raw_df, column_name, dim_base_path):
    dim_col = column_name
    id_col = f"{column_name}_id"
    dim_path = f"{dim_base_path}/dim_{column_name}"

    # 1️⃣ Try to read existing dimension table
    try:
        dim_existing = spark.read.parquet(dim_path)
    except:
        # First-time: create empty DataFrame
        schema = StructType([
            StructField(dim_col, StringType(), True),
            StructField(id_col, IntegerType(), True)
        ])
        dim_existing = spark.createDataFrame([], schema)

    # 2️⃣ Find new distinct values not in existing dim
    new_values = EQ_gold_raw_df.select(dim_col).distinct() \
                        .join(dim_existing, on=dim_col, how="left_anti")

    # 3️⃣ Assign new IDs starting after max
    if dim_existing.count() > 0:
        max_id = dim_existing.agg({id_col: "max"}).collect()[0][0]
    else:
        max_id = 0

    new_values = new_values.withColumn(
        id_col,
        row_number().over(Window.orderBy(dim_col)) + max_id
    )

    # 4️⃣ Final dimension table
    dim_final = dim_existing.unionByName(new_values)

    # 5️⃣ Save for reuse
    dim_final.write.mode("overwrite").parquet(dim_path)

    return dim_final


In [0]:
dim_columns = ["properties_type", "alert", "geometry_type"]
dim_base_path = "abfss://study@azsynapsestudy.dfs.core.windows.net/ADB/gold"

dim_tables = {}

for col_name in dim_columns:
    dim_df = handle_dimension(EQ_gold_raw_df, col_name, dim_base_path)
    dim_tables[col_name] = dim_df



In [0]:
dim_properties_type=spark.read.format("parquet").option("inferschema", "True").load("abfss://study@azsynapsestudy.dfs.core.windows.net/ADB/gold/dim_properties_type")

dim_alert=spark.read.format("parquet").option("inferschema", "True").load("abfss://study@azsynapsestudy.dfs.core.windows.net/ADB/gold/dim_alert")

dim_geometry_type=spark.read.format("parquet").option("inferschema", "True").load("abfss://study@azsynapsestudy.dfs.core.windows.net/ADB/gold/dim_geometry_type")

In [0]:


# Perform the join operations
fact_eq_df = EQ_gold_raw_df.join(dim_properties_type, on="properties_type", how="inner") \
   .join(dim_geometry_type, on="geometry_type", how="inner") \
   .join(dim_alert, on="alert", how="inner") \
   .drop("properties_type", "geometry_type", "alert","time","updated")

# Display the result
display(fact_eq_df.limit(2))

type,mag,place,tz,url,detail,felt,cdi,mmi,status,tsunami,sig,net,code,sources,types,nst,dmin,rms,gap,magType,title,latitude,longitude,depth,id,date_time,Updated_date_time,properties_type_id,geometry_type_id,alert_id
Feature,5.6,"171 km S of Mata-Utu, Wallis and Futuna",null,https://earthquake.usgs.gov/earthquakes/eventpage/usb000qt9y,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=usb000qt9y&format=geojson,null,null,3.77,Reviewed,0,482,US,b000qt9y,iscgem,",cap,losspager,moment-tensor,origin,phase-data,shakemap,",null,6.498,0.86,50,Mwc,"M 5.6 - 171 km S of Mata-Utu, Wallis and Futuna",-175.9032,-175.9032,-175.9032,usb000qt9y,2014-05-18T06:38:40.07Z,2022-05-03T16:51:39.628Z,1,1,2
Feature,6,off the west coast of northern Sumatra,null,https://earthquake.usgs.gov/earthquakes/eventpage/usb000qt4l,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=usb000qt4l&format=geojson,null,null,2.53,Reviewed,0,554,US,b000qt4l,us,",cap,losspager,moment-tensor,origin,phase-data,shakemap,",null,4.287,0.71,19,Mww,M 6.0 - off the west coast of northern Sumatra,92.7574,92.7574,92.7574,usb000qt4l,2014-05-18T01:02:32.61Z,2022-05-03T16:51:38.353Z,1,1,2


In [0]:
fact_eq_df.write.mode("append").format("parquet").save("abfss://study@azsynapsestudy.dfs.core.windows.net/ADB/gold/fact_eq")